In [3]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("E-Commerce Big Data Analytics") \
    .master("local[*]") \
    .getOrCreate()



c:\Users\MahmoudAburabeh\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
df = spark.read.csv(
    "../Data/OnlineRetail.csv",
    encoding="ISO-8859-1",
    header=True,
    inferSchema=True
)

In [5]:
df.show(5)

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|1/12/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|1/12/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|1/12/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|1/12/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|1/12/2010 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
only showing top 5 rows


In [6]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



In [7]:
print(df.columns)

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [8]:
df.describe().show()

+-------+------------------+------------------+--------------------+-----------------+---------------+-----------------+------------------+-----------+
|summary|         InvoiceNo|         StockCode|         Description|         Quantity|    InvoiceDate|        UnitPrice|        CustomerID|    Country|
+-------+------------------+------------------+--------------------+-----------------+---------------+-----------------+------------------+-----------+
|  count|            541909|            541909|              540455|           541909|         541909|           541909|            406829|     541909|
|   mean|  559965.752026781|27623.240210938104|             20713.0| 9.55224954743324|           NULL| 4.61111362608923|15287.690570239585|       NULL|
| stddev|13428.417280797195|16799.737628427698|                NULL|218.0811578502335|           NULL|96.75985306117947|1713.6003033215973|       NULL|
|    min|            536365|             10002| 4 PURPLE FLOCK D...|           -80995|1/

In [9]:
df.select("Country").distinct().show(10)

+---------+
|  Country|
+---------+
|   Sweden|
|  Germany|
|   France|
|  Belgium|
|  Finland|
|    Italy|
|     EIRE|
|Lithuania|
|   Norway|
|    Spain|
+---------+
only showing top 10 rows


In [10]:
print("Number of countries:", df.select("Country").distinct().count())

Number of countries: 38


Data Cleaning

In [11]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|       1454|       0|          0|        0|    135080|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [12]:
df_clean = df

1. Remove duplicate records

In [13]:
print("Rows:", df_clean.count())
print("Distinct rows:", df_clean.dropDuplicates().count())

Rows: 541909
Distinct rows: 536641


In [14]:
df_clean = df_clean.dropDuplicates()

In [15]:
print("Rows after removing duplicates:", df_clean.count())

Rows after removing duplicates: 536641


2. Remove transactions with missing CustomerID.


In [16]:
df_clean = df_clean.dropna(subset=["CustomerID"])

In [17]:
print("Rows after removing missing CustomerID:", df_clean.count())

Rows after removing missing CustomerID: 401604


3. Remove transactions with non-positive.


In [18]:
df_clean = df_clean.filter(
    col("Quantity") > 0
)
print("Rows after filtering for positive quantities:", df_clean.count())

df_clean = df_clean.filter(
    col("UnitPrice") > 0
)
print("Rows after filtering for positive unit prices:", df_clean.count())

df_clean = df_clean.filter(
    ~col("InvoiceNo").startswith("C")
)
print("Rows after filtering out credit notes:", df_clean.count())

Rows after filtering for positive quantities: 392732
Rows after filtering for positive unit prices: 392692
Rows after filtering out credit notes: 392692


4. Convert InvoiceDate to timestamp.


In [19]:
# Convert InvoiceDate from String to timestamp

from pyspark.sql.functions import col, expr

df_clean = df_clean.withColumn(
    "InvoiceDate_raw",
    col("InvoiceDate")
)


In [20]:
df_clean = df_clean.withColumn(
    "InvoiceDate",
    expr("""
        coalesce(
            try_to_timestamp(InvoiceDate_raw, 'M/d/yyyy HH:mm'),
            try_to_timestamp(InvoiceDate_raw, 'MM/dd/yyyy HH:mm'),
            try_to_timestamp(InvoiceDate_raw, 'dd-MM-yyyy HH:mm'),
            try_to_timestamp(InvoiceDate_raw, 'd-MM-yyyy HH:mm'),
            try_to_timestamp(InvoiceDate_raw, 'dd/MM/yyyy HH:mm'),
            try_to_timestamp(InvoiceDate_raw, 'd/M/yyyy HH:mm'),
            try_to_timestamp(InvoiceDate_raw, 'M/d/yyyy H:mm'),
            try_to_timestamp(InvoiceDate_raw, 'MM/dd/yyyy H:mm'),
            try_to_timestamp(InvoiceDate_raw, 'dd-MM-yyyy H:mm'),
            try_to_timestamp(InvoiceDate_raw, 'dd/MM/yyyy H:mm')
        )
    """)
)

In [21]:
df_clean.filter(
    col("InvoiceDate").isNull()
).select(
    "InvoiceDate_raw"
).count()

0

In [22]:
df_clean = df_clean.drop("InvoiceDate_raw")

Feature Engineering: Create TotalPrice as Quantity × UnitPrice.

New features are created from the cleaned transaction data to support
sales, product, and customer analysis.

In [23]:
from pyspark.sql.functions import col

df_clean = df_clean.withColumn(
    "TotalPrice",
    col("Quantity") * col("UnitPrice")
)

df_clean.select("Quantity", "UnitPrice", "TotalPrice").show(5)

+--------+---------+------------------+
|Quantity|UnitPrice|        TotalPrice|
+--------+---------+------------------+
|       6|     5.45|              32.7|
|      24|     0.55|13.200000000000001|
|       1|     1.65|              1.65|
|       2|     1.65|               3.3|
|       2|     0.85|               1.7|
+--------+---------+------------------+
only showing top 5 rows


In [24]:
from pyspark.sql.functions import (
    year,
    month,
    dayofmonth,
    hour,
    col
)

In [25]:
# Create new columns for Year, Month, Day, and Hour from the InvoiceDate column

df_clean = df_clean.withColumn(
    "Year",
    year("InvoiceDate")
)

df_clean = df_clean.withColumn(
    "Month",
    month("InvoiceDate")
)

df_clean = df_clean.withColumn(
    "Day",
    dayofmonth("InvoiceDate")
)


df_clean = df_clean.withColumn(
    "Hour",
    hour("InvoiceDate")
)


In [26]:
df_clean.select(
    "InvoiceDate",
    "Year",
    "Month",
    "Day",
    "Hour"
).show(10, False)

+-------------------+----+-----+---+----+
|InvoiceDate        |Year|Month|Day|Hour|
+-------------------+----+-----+---+----+
|2010-01-12 10:03:00|2010|1    |12 |10  |
|2010-01-12 11:29:00|2010|1    |12 |11  |
|2010-01-12 11:49:00|2010|1    |12 |11  |
|2010-01-12 11:49:00|2010|1    |12 |11  |
|2010-01-12 12:23:00|2010|1    |12 |12  |
|2010-01-12 13:24:00|2010|1    |12 |13  |
|2010-01-12 14:03:00|2010|1    |12 |14  |
|2010-02-12 11:41:00|2010|2    |12 |11  |
|2010-02-12 13:06:00|2010|2    |12 |13  |
|2010-02-12 13:08:00|2010|2    |12 |13  |
+-------------------+----+-----+---+----+
only showing top 10 rows


Sales Analysis

In this section, I analyze the cleaned transaction data to identify
overall sales performance, revenue trends, and key business metrics.

In [27]:
from pyspark.sql.functions import sum, countDistinct

sales_summary = df_clean.agg(
    sum("TotalPrice").alias("TotalRevenue"),
    countDistinct("InvoiceNo").alias("TotalOrders"),
    countDistinct("CustomerID").alias("TotalCustomers"),
    countDistinct("StockCode").alias("TotalProducts")
)

print("Sales Summary:")
sales_summary.show()

Sales Summary:
+-----------------+-----------+--------------+-------------+
|     TotalRevenue|TotalOrders|TotalCustomers|TotalProducts|
+-----------------+-----------+--------------+-------------+
|8887208.894000202|      18532|          4338|         3665|
+-----------------+-----------+--------------+-------------+



Calculate Average of Orders

In [28]:
# order value per invoice
order_values = df_clean.groupBy("InvoiceNo").agg(
    sum("TotalPrice").alias("OrderValue")
)
print("Order Values:")
order_values.show(20, False)

Order Values:
+---------+------------------+
|InvoiceNo|OrderValue        |
+---------+------------------+
|537691   |310.57            |
|538879   |338.98            |
|540499   |348.19999999999993|
|538517   |278.98            |
|539630   |751.0             |
|540540   |169.35            |
|543641   |186.34            |
|544303   |161.95000000000005|
|541518   |2919.5099999999993|
|541783   |728.1600000000001 |
|540976   |919.98            |
|548542   |198.20000000000002|
|550469   |178.41            |
|553390   |466.28            |
|552852   |414.5             |
|550831   |171.20000000000002|
|553470   |5410.0599999999995|
|553738   |305.36            |
|552277   |312.28000000000003|
|550617   |376.0             |
+---------+------------------+
only showing top 20 rows


In [29]:
from pyspark.sql.functions import avg

average_order_value = order_values.agg(
    avg("OrderValue").alias("AverageOrderValue")
)

print("Average Order Value:", average_order_value.collect()[0]["AverageOrderValue"])

Average Order Value: 479.56016047917313


Total Revenue per (Month)

In [30]:
from pyspark.sql.functions import sum, col

monthly_revenue = df_clean.groupBy(
    "Year",
    "Month"
).agg(
    sum("TotalPrice").alias("Revenue")
).orderBy(
    "Year",
    "Month"
)


print("Monthly Revenue:")
monthly_revenue.show(15, False)


Monthly Revenue:
+----+-----+------------------+
|Year|Month|Revenue           |
+----+-----+------------------+
|2010|1    |46192.48999999999 |
|2010|2    |47197.57          |
|2010|3    |23876.630000000005|
|2010|5    |31361.280000000002|
|2010|6    |31009.329999999994|
|2010|7    |53730.96000000001 |
|2010|8    |39094.200000000004|
|2010|9    |38193.909999999996|
|2010|10   |33488.380000000005|
|2010|12   |226277.98000000024|
|2011|1    |600330.0100000006 |
|2011|2    |485607.660000001  |
|2011|3    |644633.570000001  |
|2011|4    |601577.8610000008 |
|2011|5    |717482.9000000003 |
+----+-----+------------------+
only showing top 15 rows


In [31]:
# Top 10 Months by Revenue

print("Top 10 Months by Revenue:")
monthly_revenue.orderBy(
    col("Revenue").desc()
).show(10, False)

Top 10 Months by Revenue:
+----+-----+------------------+
|Year|Month|Revenue           |
+----+-----+------------------+
|2011|9    |1093319.7020000003|
|2011|11   |1039157.8099999988|
|2011|10   |901925.9499999988 |
|2011|7    |733983.1810000007 |
|2011|5    |717482.9000000003 |
|2011|6    |687078.5400000007 |
|2011|3    |644633.570000001  |
|2011|8    |607009.5900000008 |
|2011|4    |601577.8610000008 |
|2011|1    |600330.0100000006 |
+----+-----+------------------+
only showing top 10 rows


Country Revenue

In [32]:
country_revenue = df_clean.groupBy(
    "Country"
).agg(
    sum("TotalPrice").alias("Revenue")
).orderBy(
    col("Revenue").desc()
)

print("Country Revenue:")
country_revenue.show(10, truncate=False)

Country Revenue:
+--------------+------------------+
|Country       |Revenue           |
+--------------+------------------+
|United Kingdom|7285024.64400018  |
|Netherlands   |285446.33999999997|
|EIRE          |265262.46         |
|Germany       |228678.3999999999 |
|France        |208934.31         |
|Australia     |138453.81         |
|Spain         |61558.56          |
|Switzerland   |56443.950000000004|
|Belgium       |41196.33999999999 |
|Sweden        |38367.829999999994|
+--------------+------------------+
only showing top 10 rows


Product Analysis

In [33]:
# top 10 products by revenue

product_revenue = df_clean.groupBy(
    "StockCode",
    "Description"
).agg(
    sum("TotalPrice").alias("Revenue")
).orderBy(
    col("Revenue").desc()
)

print("Product Revenue:")
product_revenue.show(10, truncate=False)

Product Revenue:
+---------+----------------------------------+-----------------+
|StockCode|Description                       |Revenue          |
+---------+----------------------------------+-----------------+
|23843    |PAPER CRAFT , LITTLE BIRDIE       |168469.6         |
|22423    |REGENCY CAKESTAND 3 TIER          |142264.75        |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|100392.1         |
|85099B   |JUMBO BAG RED RETROSPOT           |85040.54000000007|
|23166    |MEDIUM CERAMIC TOP STORAGE JAR    |81416.73000000001|
|POST     |POSTAGE                           |77803.95999999999|
|47566    |PARTY BUNTING                     |68785.23000000003|
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |56413.03         |
|M        |Manual                            |53419.93         |
|23084    |RABBIT NIGHT LIGHT                |51251.23999999999|
+---------+----------------------------------+-----------------+
only showing top 10 rows


In [34]:
product_quantity = df_clean.groupBy(
    "StockCode",
    "Description"
).agg(
    sum("Quantity").alias("UnitsSold")
).orderBy(
    col("UnitsSold").desc()
)

print("Product Quantity Sold:")
product_quantity.show(10, truncate=False)

Product Quantity Sold:
+---------+----------------------------------+---------+
|StockCode|Description                       |UnitsSold|
+---------+----------------------------------+---------+
|23843    |PAPER CRAFT , LITTLE BIRDIE       |80995    |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR    |77916    |
|84077    |WORLD WAR 2 GLIDERS ASSTD DESIGNS |54319    |
|85099B   |JUMBO BAG RED RETROSPOT           |46078    |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|36706    |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |35263    |
|21212    |PACK OF 72 RETROSPOT CAKE CASES   |33670    |
|22197    |POPCORN HOLDER                    |30919    |
|23084    |RABBIT NIGHT LIGHT                |27153    |
|22492    |MINI PAINT SET VINTAGE            |26076    |
+---------+----------------------------------+---------+
only showing top 10 rows


Customer Analysis

In [35]:
# Customer Revenue

customer_revenue = df_clean.groupBy(
    "CustomerID"
).agg(
    sum("TotalPrice").alias("TotalSpent")
).orderBy(
    col("TotalSpent").desc()
)

print("Customer Revenue:")
customer_revenue.show(10)

Customer Revenue:
+----------+------------------+
|CustomerID|        TotalSpent|
+----------+------------------+
|     14646|         280206.02|
|     18102|259657.30000000002|
|     17450|         194390.79|
|     16446|          168472.5|
|     14911|143711.16999999998|
|     12415|124914.53000000001|
|     14156|         117210.08|
|     17511| 91062.38000000002|
|     16029|          80850.84|
|     12346|           77183.6|
+----------+------------------+
only showing top 10 rows


In [36]:
customer_orders = df_clean.groupBy(
    "CustomerID"
).agg(
    countDistinct("InvoiceNo").alias("Orders")
).orderBy(
    col("Orders").desc()
)

print("Customer Orders:")
customer_orders.show(10)

Customer Orders:
+----------+------+
|CustomerID|Orders|
+----------+------+
|     12748|   209|
|     14911|   201|
|     17841|   124|
|     13089|    97|
|     14606|    93|
|     15311|    91|
|     12971|    86|
|     14646|    73|
|     16029|    63|
|     13408|    62|
+----------+------+
only showing top 10 rows


In [37]:
hourly_sales = df_clean.groupBy(
    "Hour"
).agg(
    sum("TotalPrice").alias("Revenue"),
    countDistinct("InvoiceNo").alias("Orders")
).orderBy(
    "Hour"
)

print("Hourly Sales:")
hourly_sales.show(24)

Hourly Sales:
+----+------------------+------+
|Hour|           Revenue|Orders|
+----+------------------+------+
|   6|              4.25|     1|
|   7|          31059.21|    29|
|   8|281997.79000000015|   555|
|   9| 842392.3409999998|  1393|
|  10|       1259267.591|  2226|
|  11|1101177.6000000003|  2277|
|  12|1373695.3900000006|  3130|
|  13|1168724.1999999993|  2636|
|  14| 991992.8210000005|  2274|
|  15|         963559.68|  2037|
|  16|         467380.56|  1100|
|  17|233811.59100000001|   544|
|  18|104744.99000000002|   169|
|  19|           48568.4|   144|
|  20|          18832.48|    18|
+----+------------------+------+



In [38]:
daily_sales = df_clean.groupBy(
    "Day"
).agg(
    sum("TotalPrice").alias("Revenue")
).orderBy(
    "Day"
)

print("Daily Sales:")
daily_sales.show(31)

Daily Sales:
+---+------------------+
|Day|           Revenue|
+---+------------------+
|  1|207336.10000000006|
|  2|         169331.84|
|  3|         210436.85|
|  4|195226.46000000008|
|  5| 281143.6800000001|
|  6|         277611.29|
|  7|         187289.01|
|  8|293785.75000000006|
|  9|278482.73000000004|
| 10| 424713.3600000006|
| 11| 461668.4700000009|
| 12| 878437.5400000012|
| 13|251257.43000000002|
| 14|336979.27000000037|
| 15| 304031.7310000001|
| 16| 278095.9400000001|
| 17| 332207.2500000003|
| 18| 367310.1699999999|
| 19|249359.43999999997|
| 20|362244.59000000026|
| 21|325268.91000000015|
| 22| 249685.5099999999|
| 23|248552.10000000012|
| 24|266446.33999999997|
| 25|227452.11100000003|
| 26|        183508.712|
| 27|223455.75000000015|
| 28|296705.82000000024|
| 29|167577.82999999993|
| 30| 202399.0499999999|
| 31|         149207.86|
+---+------------------+



RFM --> Recency - Frequency - Monetary

In [39]:
from pyspark.sql.functions import max, datediff, lit

reference_date = df_clean.agg(
    max("InvoiceDate")
).collect()[0][0]

rfm = df_clean.groupBy(
    "CustomerID"
).agg(
    datediff(
        lit(reference_date),
        max("InvoiceDate")
    ).alias("Recency"),

    countDistinct("InvoiceNo").alias("Frequency"),

    sum("TotalPrice").alias("Monetary")
)

print("RFM Analysis:")
rfm.show(10)

RFM Analysis:
+----------+-------+---------+------------------+
|CustomerID|Recency|Frequency|          Monetary|
+----------+-------+---------+------------------+
|     17389|      9|       34|          31833.68|
|     14450|      4|        3|            483.25|
|     15727|      9|        7|           5159.06|
|     15790|     11|        1|218.74999999999997|
|     13285|     24|        4|           2709.12|
|     14570|    251|        2|            218.06|
|     16574|     72|        1|            451.44|
|     15619|     11|        1|336.40000000000003|
|     13832|     20|        1|52.199999999999996|
|     14465|     69|        3|           1038.44|
+----------+-------+---------+------------------+
only showing top 10 rows


RFM Segmentation

In [40]:
from pyspark.sql.window import Window
from pyspark.sql.functions import ntile, col

# Recency: 
recency_window = Window.orderBy(col("Recency").asc())

# Frequency: 
frequency_window = Window.orderBy(col("Frequency").desc())

# Monetary: 
monetary_window = Window.orderBy(col("Monetary").desc())

rfm_scored = (
    rfm
    .withColumn("R_Score", ntile(5).over(recency_window))
    .withColumn("F_Score", ntile(5).over(frequency_window))
    .withColumn("M_Score", ntile(5).over(monetary_window))
)

print("RFM Scored:")
rfm_scored.show(10)


# R_score: 1 (most recent) to 5 (least recent)
# F_score: 1 (least frequent) to 5 (most frequent)
# M_score: 1 (lowest monetary) to 5 (highest monetary)

RFM Scored:
+----------+-------+---------+------------------+-------+-------+-------+
|CustomerID|Recency|Frequency|          Monetary|R_Score|F_Score|M_Score|
+----------+-------+---------+------------------+-------+-------+-------+
|     14646|      2|       73|         280206.02|      1|      1|      1|
|     18102|     12|       60|259657.30000000002|      1|      1|      1|
|     17450|      3|       46|194390.78999999998|      1|      1|      1|
|     16446|     89|        2|          168472.5|      4|      3|      1|
|     14911|      0|      201|         143711.17|      1|      1|      1|
|     12415|     25|       21|         124914.53|      2|      1|      1|
|     14156|      0|       55|117210.08000000002|      1|      1|      1|
|     17511|      6|       31|          91062.38|      1|      1|      1|
|     16029|     30|       63|          80850.84|      2|      1|      1|
|     12346|    326|        1|           77183.6|      5|      5|      1|
+----------+-------+------

Segments

In [41]:
from pyspark.sql.functions import when

rfm_segmented = rfm_scored.withColumn(
    "Segment",
    when(
        (col("R_Score") >= 4) &
        (col("F_Score") >= 4) &
        (col("M_Score") >= 4),
        "Champions"
    )
    .when(
        (col("R_Score") >= 3) &
        (col("F_Score") >= 3),
        "Loyal Customers"
    )
    .when(
        (col("R_Score") <= 2) &
        (col("F_Score") >= 3),
        "At Risk"
    )
    .when(
        (col("R_Score") >= 4) &
        (col("F_Score") <= 2),
        "New Customers"
    )
    .otherwise(
        "Hibernating"
    )
)

In [42]:
segment_distribution = rfm_segmented.groupBy(
    "Segment"
).count().orderBy(
    col("count").desc()
)

print("Segment Distribution:")
segment_distribution.show()

Segment Distribution:
+---------------+-----+
|        Segment|count|
+---------------+-----+
|    Hibernating| 1531|
|Loyal Customers| 1166|
|      Champions|  926|
|        At Risk|  510|
|  New Customers|  205|
+---------------+-----+



Percentage of Segments

In [43]:
from pyspark.sql.functions import sum, count, col

total_customer_count = rfm_segmented.count()

segment_distribution = (
    rfm_segmented
    .groupBy("Segment")
    .count()
    .withColumn(
        "Percentage",
        col("count") / total_customer_count * 100
    )
    .orderBy(
        col("count").desc()
    )
)

print("Segment Distribution with Percentage:")
segment_distribution.show()

Segment Distribution with Percentage:
+---------------+-----+------------------+
|        Segment|count|        Percentage|
+---------------+-----+------------------+
|    Hibernating| 1531| 35.29276164130936|
|Loyal Customers| 1166|26.878745965882896|
|      Champions|  926|21.346242508068233|
|        At Risk|  510|11.756569847856154|
|  New Customers|  205|4.7256800368833565|
+---------------+-----+------------------+



Every Segment  

In [44]:
segment_revenue = (
    rfm_segmented
    .groupBy("Segment")
    .agg(
        sum("Monetary").alias("Revenue"),
        count("*").alias("Customers")
    )
    .orderBy(
        col("Revenue").desc()
    )
)

print("Segment Revenue:")
segment_revenue.show()

Segment Revenue:
+---------------+------------------+---------+
|        Segment|           Revenue|Customers|
+---------------+------------------+---------+
|    Hibernating| 6769097.539999994|     1531|
|Loyal Customers|1202438.6630000006|     1166|
|  New Customers| 437059.7310000004|      205|
|        At Risk| 264239.6400000002|      510|
|      Champions|214373.32000000004|      926|
+---------------+------------------+---------+



Avg every Customer in spending 

In [45]:
from pyspark.sql.functions import avg

segment_analysis = (
    rfm_segmented
    .groupBy("Segment")
    .agg(
        count("*").alias("Customers"),
        avg("Recency").alias("AvgRecency"),
        avg("Frequency").alias("AvgFrequency"),
        avg("Monetary").alias("AvgMonetary"),
        sum("Monetary").alias("TotalRevenue")
    )
    .orderBy(
        col("TotalRevenue").desc()
    )
)

print("Segment Analysis:")
segment_analysis.show()

Segment Analysis:
+---------------+---------+------------------+------------------+------------------+------------------+
|        Segment|Customers|        AvgRecency|      AvgFrequency|       AvgMonetary|      TotalRevenue|
+---------------+---------+------------------+------------------+------------------+------------------+
|    Hibernating|     1531|25.615937295885043|   8.7472240365774| 4421.356982364464| 6769097.539999994|
|Loyal Customers|     1166|125.07204116638079|1.8559176672384219|1031.2509974271018|1202438.6630000006|
|  New Customers|      205|143.89756097560976| 5.634146341463414|  2131.99868780488| 437059.7310000004|
|        At Risk|      510| 20.88235294117647| 1.535294117647059|  518.116941176471| 264239.6400000002|
|      Champions|      926| 251.0475161987041|1.1209503239740821| 231.5046652267819|214373.32000000004|
+---------------+---------+------------------+------------------+------------------+------------------+



In [46]:
# ============================================================
# Export Results for Streamlit Dashboard
# ============================================================

# 1. KPIs
kpis = df_clean.agg(
    sum("TotalPrice").alias("TotalRevenue"),
    countDistinct("InvoiceNo").alias("TotalOrders"),
    countDistinct("CustomerID").alias("TotalCustomers"),
    countDistinct("StockCode").alias("TotalProducts")
)

kpis.toPandas().to_csv(
    "../dashboard/kpis.csv",
    index=False
)


# 2. Monthly Revenue
monthly_revenue.toPandas().to_csv(
    "../dashboard/monthly_revenue.csv",
    index=False
)


# 3. Country Revenue
country_revenue.toPandas().to_csv(
    "../dashboard/country_revenue.csv",
    index=False
)


# 4. Product Revenue
product_revenue.toPandas().to_csv(
    "../dashboard/product_revenue.csv",
    index=False
)


# 5. Hourly Sales
hourly_sales.toPandas().to_csv(
    "../dashboard/hourly_sales.csv",
    index=False
)


# 6. Customer Segment Distribution
segment_distribution.toPandas().to_csv(
    "../dashboard/segment_distribution.csv",
    index=False
)


# 7. Revenue by Customer Segment
segment_revenue.toPandas().to_csv(
    "../dashboard/segment_revenue.csv",
    index=False
)

# 8. RFM Segmentation
rfm_segmented.toPandas().to_csv(
		"../dashboard/rfm_segmentation.csv",
		index=False
)


print("All dashboard files exported successfully.")

c:\Users\MahmoudAburabeh\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\MahmoudAburabeh\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\MahmoudAburabeh\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\MahmoudAburabeh\AppData\Local\Programs\Python\Python314\Lib\s

All dashboard files exported successfully.


In [47]:
rfm_segmented.toPandas().to_csv(
    "../dashboard/rfm_customers.csv",
    index=False
)

print("RFM customers file exported successfully.")

RFM customers file exported successfully.


c:\Users\MahmoudAburabeh\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [58]:
df_clean.toPandas().to_csv(
    "../dashboard/data_clean.csv",
    index=False
)

c:\Users\MahmoudAburabeh\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\MahmoudAburabeh\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
